# 🏛️ Project 1: AI-Driven Citizen Grievance & Sentiment Analysis
## Week 4 — API Development, Evaluation & Final Delivery
**Infotact Technical Internship Program**

---
### 📌 Week 4 Goals:
- [ ] Evaluate classification model with Confusion Matrix & Classification Report
- [ ] Evaluate sentiment model with Macro F1-Score
- [ ] Serialize (save) trained models using joblib
- [ ] Build a FastAPI application with a `/predict` endpoint
- [ ] Test the API with sample JSON payloads
- [ ] Final GitHub cleanup and documentation

---
**Dataset columns used:**
- `product` → Department label (credit_card, retail_banking, credit_reporting, mortgages_and_loans, debt_collection)
- `processed_text` → Cleaned & lemmatized complaint text
- `narrative` → Original raw complaint text
---

## 📦 Step 1: Install & Import Libraries

In [1]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib fastapi uvicorn nest-asyncio


   ---------- ----------------------------- 1/4 [uvicorn]
   ---------- ----------------------------- 1/4 [uvicorn]
   ---------- ----------------------------- 1/4 [uvicorn]
   ---------- ----------------------------- 1/4 [uvicorn]
   ---------- ----------------------------- 1/4 [uvicorn]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries loaded!')

KeyboardInterrupt: 

---
## 📂 Step 2: Load Processed Dataset

In [ ]:
# Load dataset
# Upload processed_grievances.csv to Colab before running this cell
df = pd.read_csv('processed_grievances.csv')

# Rename columns to match project terminology
df = df.rename(columns={'product': 'department'})

# Drop rows with missing processed_text
df = df.dropna(subset=['processed_text'])
df['processed_text'] = df['processed_text'].astype(str)

print(f'✅ Dataset loaded: {df.shape[0]:,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nDepartment distribution:')
print(df['department'].value_counts())

In [ ]:
# Use a stratified sample for faster training (use full data if GPU available)
df_sample = df.groupby('department', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 3000), random_state=42)
).reset_index(drop=True)

print(f'✅ Sampled dataset: {len(df_sample):,} rows')
print(df_sample['department'].value_counts())

---
## 🤖 Step 3: Add Synthetic Sentiment Labels
> Since the dataset doesn't have sentiment labels, we'll assign them based on urgency keywords — a common real-world NLP technique called rule-based pre-labeling.

In [ ]:
def assign_sentiment(text):
    """Assign sentiment based on urgency keywords in the complaint text."""
    text = str(text).lower()
    
    critical_keywords = [
        'fraud', 'stolen', 'illegal', 'lawsuit', 'attorney', 'court',
        'emergency', 'urgent', 'immediately', 'criminal', 'scam', 'harassment'
    ]
    negative_keywords = [
        'wrong', 'error', 'problem', 'issue', 'complaint', 'denied',
        'unfair', 'incorrect', 'failed', 'refuse', 'never', 'worst',
        'terrible', 'horrible', 'angry', 'frustrated', 'disappoint'
    ]
    positive_keywords = [
        'thank', 'great', 'good', 'excellent', 'resolved', 'happy',
        'satisfied', 'appreciate', 'helpful', 'perfect', 'wonderful'
    ]
    
    if any(word in text for word in critical_keywords):
        return 'Critical'
    elif any(word in text for word in negative_keywords):
        return 'Negative'
    elif any(word in text for word in positive_keywords):
        return 'Positive'
    else:
        return 'Neutral'

df_sample['sentiment'] = df_sample['narrative'].apply(assign_sentiment)

print('✅ Sentiment labels assigned!')
print(df_sample['sentiment'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_counts = df_sample['department'].value_counts()
axes[0].bar(dept_counts.index, dept_counts.values, color=sns.color_palette('Set2'))
axes[0].set_title('Complaint Count by Department', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylabel('Count')

sent_counts = df_sample['sentiment'].value_counts()
colors = {'Critical': '#e74c3c', 'Negative': '#e67e22', 'Neutral': '#3498db', 'Positive': '#2ecc71'}
axes[1].pie(
    sent_counts.values,
    labels=sent_counts.index,
    autopct='%1.1f%%',
    colors=[colors.get(s, 'grey') for s in sent_counts.index],
    startangle=140
)
axes[1].set_title('Sentiment Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('week4_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🏗️ Step 4: Train-Test Split

In [ ]:
X = df_sample['processed_text']
y_dept = df_sample['department']
y_sent = df_sample['sentiment']

# Split for department classifier
X_train, X_test, y_dept_train, y_dept_test = train_test_split(
    X, y_dept, test_size=0.2, random_state=42, stratify=y_dept
)

# Split for sentiment classifier (same indices)
_, _, y_sent_train, y_sent_test = train_test_split(
    X, y_sent, test_size=0.2, random_state=42, stratify=y_sent
)

print(f'✅ Train set: {len(X_train):,} samples')
print(f'✅ Test set:  {len(X_test):,} samples')

---
## 🎯 Step 5: Department Classification Model (TF-IDF + Logistic Regression)

In [ ]:
print('🔄 Training Department Classifier...')

# Build pipeline
dept_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver='lbfgs',
        multi_class='multinomial',
        random_state=42
    ))
])

dept_pipeline.fit(X_train, y_dept_train)
y_dept_pred = dept_pipeline.predict(X_test)

dept_acc = accuracy_score(y_dept_test, y_dept_pred)
dept_f1  = f1_score(y_dept_test, y_dept_pred, average='macro')

print(f'\n✅ Department Classifier Results:')
print(f'   Accuracy:   {dept_acc:.4f} ({dept_acc*100:.2f}%)')
print(f'   Macro F1:   {dept_f1:.4f}')

---
## 📊 Step 6: Department Classifier Evaluation

In [ ]:
# Classification Report
print('='*60)
print('DEPARTMENT CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_dept_test, y_dept_pred))

In [ ]:
# Confusion Matrix
dept_labels = sorted(y_dept.unique())
cm_dept = confusion_matrix(y_dept_test, y_dept_pred, labels=dept_labels)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dept, display_labels=dept_labels)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=30)
ax.set_title('Department Classifier — Confusion Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('week4_dept_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
# Per-class F1 bar chart
report_dict = classification_report(y_dept_test, y_dept_pred, output_dict=True)
classes = [k for k in report_dict if k not in ['accuracy', 'macro avg', 'weighted avg']]
f1_scores = [report_dict[c]['f1-score'] for c in classes]

plt.figure(figsize=(10, 5))
bars = plt.bar(classes, f1_scores, color=sns.color_palette('Set2', len(classes)), edgecolor='white')
plt.axhline(y=dept_f1, color='red', linestyle='--', linewidth=1.5, label=f'Macro F1 = {dept_f1:.3f}')
plt.title('F1-Score per Department Class', fontsize=14, fontweight='bold')
plt.xlabel('Department')
plt.ylabel('F1-Score')
plt.ylim(0, 1.1)
plt.legend()
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('week4_dept_f1_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ F1 chart saved!')

---
## 😊 Step 7: Sentiment Analysis Model

In [ ]:
print('🔄 Training Sentiment Classifier...')

sent_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',   # handles class imbalance
        solver='lbfgs',
        multi_class='multinomial',
        random_state=42
    ))
])

sent_pipeline.fit(X_train, y_sent_train)
y_sent_pred = sent_pipeline.predict(X_test)

sent_acc = accuracy_score(y_sent_test, y_sent_pred)
sent_f1  = f1_score(y_sent_test, y_sent_pred, average='macro')

print(f'\n✅ Sentiment Classifier Results:')
print(f'   Accuracy:   {sent_acc:.4f} ({sent_acc*100:.2f}%)')
print(f'   Macro F1:   {sent_f1:.4f}')

In [ ]:
# Sentiment Classification Report
print('='*60)
print('SENTIMENT CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_sent_test, y_sent_pred))

In [ ]:
# Sentiment Confusion Matrix
sent_labels = sorted(y_sent.unique())
cm_sent = confusion_matrix(y_sent_test, y_sent_pred, labels=sent_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sent, display_labels=sent_labels)
disp.plot(ax=ax, cmap='Reds', colorbar=True)
ax.set_title('Sentiment Classifier — Confusion Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('week4_sent_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sentiment confusion matrix saved!')

---
## 🔢 Step 8: Priority Score Assignment

In [ ]:
PRIORITY_MAP = {
    'Critical': 1.0,
    'Negative': 0.65,
    'Neutral':  0.35,
    'Positive': 0.10
}

def get_priority_score(sentiment: str, confidence: float) -> float:
    """Compute weighted priority score from sentiment class and model confidence."""
    base_score = PRIORITY_MAP.get(sentiment, 0.35)
    return round(base_score * confidence, 4)

# Demo on test set
sent_probs = sent_pipeline.predict_proba(X_test)
sent_classes = sent_pipeline.classes_

sample_results = []
for i in range(5):
    pred_sentiment  = y_sent_pred[i]
    confidence      = sent_probs[i].max()
    priority_score  = get_priority_score(pred_sentiment, confidence)
    pred_dept       = y_dept_pred[i]
    sample_results.append({
        'text_snippet'    : X_test.iloc[i][:60] + '...',
        'dept_predicted'  : pred_dept,
        'sentiment'       : pred_sentiment,
        'confidence'      : round(float(confidence), 4),
        'priority_score'  : priority_score
    })

results_df = pd.DataFrame(sample_results)
print('✅ Sample Predictions with Priority Scores:')
print(results_df.to_string(index=False))

---
## 💾 Step 9: Save (Serialize) Models
> **Note:** Add `*.pkl` to your `.gitignore` — never push model files to GitHub.

In [ ]:
os.makedirs('models', exist_ok=True)

# Save department pipeline
joblib.dump(dept_pipeline, 'models/dept_classifier.pkl')
print('✅ Department model saved → models/dept_classifier.pkl')

# Save sentiment pipeline
joblib.dump(sent_pipeline, 'models/sent_classifier.pkl')
print('✅ Sentiment model saved → models/sent_classifier.pkl')

# Save priority map
with open('models/priority_map.json', 'w') as f:
    json.dump(PRIORITY_MAP, f, indent=2)
print('✅ Priority map saved → models/priority_map.json')

# Verify files
print('\n📂 Saved model files:')
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}')
    print(f'   {f}  ({size/1024:.1f} KB)')

---
## 🌐 Step 10: FastAPI Application
> Run the cell below to write `main.py` — then start the server in a terminal with `uvicorn main:app --reload`

In [ ]:
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import json
import re

# ── Load models ────────────────────────────────────────────────
dept_model = joblib.load("models/dept_classifier.pkl")
sent_model = joblib.load("models/sent_classifier.pkl")

with open("models/priority_map.json") as f:
    PRIORITY_MAP = json.load(f)

app = FastAPI(
    title="Citizen Grievance & Sentiment API",
    description="AI-powered complaint routing and urgency scoring system",
    version="1.0.0"
)

# ── Request & Response Schemas ─────────────────────────────────
class ComplaintRequest(BaseModel):
    complaint_text: str

class PredictionResponse(BaseModel):
    original_text:   str
    department:      str
    dept_confidence: float
    sentiment:       str
    sent_confidence: float
    priority_score:  float
    action:          str

# ── Preprocessing ──────────────────────────────────────────────
def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"[^a-z\\s]", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def get_action(sentiment: str) -> str:
    return {
        "Critical": "🚨 IMMEDIATE escalation required — dispatch within 1 hour",
        "Negative": "⚠️  High priority — respond within 24 hours",
        "Neutral":  "📋 Standard queue — respond within 3 working days",
        "Positive": "✅ Acknowledgement only — log and close"
    }.get(sentiment, "📋 Standard queue")

# ── Endpoints ──────────────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "Citizen Grievance API is running!", "status": "healthy"}

@app.get("/health")
def health():
    return {"status": "ok", "models_loaded": True}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: ComplaintRequest):
    if not request.complaint_text.strip():
        raise HTTPException(status_code=400, detail="complaint_text cannot be empty")

    cleaned = preprocess(request.complaint_text)

    # Department prediction
    dept_probs    = dept_model.predict_proba([cleaned])[0]
    dept_label    = dept_model.classes_[dept_probs.argmax()]
    dept_conf     = round(float(dept_probs.max()), 4)

    # Sentiment prediction
    sent_probs    = sent_model.predict_proba([cleaned])[0]
    sent_label    = sent_model.classes_[sent_probs.argmax()]
    sent_conf     = round(float(sent_probs.max()), 4)

    # Priority score
    base_score    = PRIORITY_MAP.get(sent_label, 0.35)
    priority      = round(base_score * sent_conf, 4)

    return PredictionResponse(
        original_text   = request.complaint_text,
        department      = dept_label,
        dept_confidence = dept_conf,
        sentiment       = sent_label,
        sent_confidence = sent_conf,
        priority_score  = priority,
        action          = get_action(sent_label)
    )
'''

with open('main.py', 'w') as f:
    f.write(fastapi_code)

print('✅ main.py written!')
print('\n🚀 To start the API server, run in terminal:')
print('   uvicorn main:app --reload')
print('\n📖 Then open in browser:')
print('   http://127.0.0.1:8000/docs  ← Interactive Swagger UI')

---
## 🧪 Step 11: Test the API in Colab (without terminal)

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
import requests

nest_asyncio.apply()

# Import app from main.py
import importlib.util, sys
spec = importlib.util.spec_from_file_location('main', 'main.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
app  = mod.app

# Start server in background thread
def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='error')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print('✅ FastAPI server started at http://127.0.0.1:8000')

In [ ]:
# Test sample complaints
BASE_URL = 'http://127.0.0.1:8000'

test_complaints = [
    "Someone fraudulently opened a credit card in my name and made unauthorized charges. This is an emergency!",
    "My mortgage payment was incorrectly reported as late even though I paid on time.",
    "Debt collectors keep calling me multiple times a day which is harassment.",
    "My credit score dropped by 100 points due to an error on my credit report.",
    "The bank resolved my issue quickly and the representative was very helpful."
]

print('=' * 80)
print('API TEST RESULTS')
print('=' * 80)

for i, complaint in enumerate(test_complaints, 1):
    response = requests.post(
        f'{BASE_URL}/predict',
        json={'complaint_text': complaint}
    )
    result = response.json()
    
    print(f'\n📝 Test {i}: {complaint[:70]}...')
    print(f'   🏢 Department    : {result["department"]} (confidence: {result["dept_confidence"]})')
    print(f'   😤 Sentiment     : {result["sentiment"]} (confidence: {result["sent_confidence"]})')
    print(f'   🎯 Priority Score: {result["priority_score"]}')
    print(f'   📋 Action        : {result["action"]}')
    print('-' * 80)

---
## 📈 Step 12: Model Performance Summary Dashboard

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Week 4 — Model Performance Summary', fontsize=16, fontweight='bold', y=1.02)

# Department model metrics
dept_report = classification_report(y_dept_test, y_dept_pred, output_dict=True)
dept_classes = [k for k in dept_report if k not in ['accuracy', 'macro avg', 'weighted avg']]
dept_f1s = [dept_report[c]['f1-score'] for c in dept_classes]

bars1 = axes[0].barh(dept_classes, dept_f1s, color=sns.color_palette('Blues_d', len(dept_classes)))
axes[0].axvline(x=dept_f1, color='red', linestyle='--', label=f'Macro F1={dept_f1:.3f}')
axes[0].set_title('Department Classifier — F1 per Class', fontweight='bold')
axes[0].set_xlabel('F1-Score')
axes[0].set_xlim(0, 1.1)
axes[0].legend()
for bar, score in zip(bars1, dept_f1s):
    axes[0].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{score:.3f}', va='center', fontsize=9)

# Sentiment model metrics
sent_report = classification_report(y_sent_test, y_sent_pred, output_dict=True)
sent_classes = [k for k in sent_report if k not in ['accuracy', 'macro avg', 'weighted avg']]
sent_f1s = [sent_report[c]['f1-score'] for c in sent_classes]

bars2 = axes[1].barh(sent_classes, sent_f1s, color=sns.color_palette('Reds_d', len(sent_classes)))
axes[1].axvline(x=sent_f1, color='blue', linestyle='--', label=f'Macro F1={sent_f1:.3f}')
axes[1].set_title('Sentiment Classifier — F1 per Class', fontweight='bold')
axes[1].set_xlabel('F1-Score')
axes[1].set_xlim(0, 1.1)
axes[1].legend()
for bar, score in zip(bars2, sent_f1s):
    axes[1].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{score:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('week4_performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Performance summary saved!')

---
## 📋 Step 13: Final GitHub Checklist

In [ ]:
gitignore_content = """# Data files — never push raw data to GitHub
*.csv
*.xlsx
*.json

# Trained model weights
*.pkl
*.h5
*.joblib

# Environment
.env
__pycache__/
*.pyc
.ipynb_checkpoints/

# OS files
.DS_Store
Thumbs.db
"""

with open('.gitignore', 'w') as f:
    f.write(gitignore_content)

print('✅ .gitignore created!')

readme_content = """# Project 1: AI-Driven Citizen Grievance & Sentiment Analysis System
### Infotact Technical Internship Program

## 📌 Overview
An NLP system that automatically routes citizen complaints to government departments
and assigns urgency priority scores using sentiment analysis.

## 🏗️ Architecture
- **Department Classifier**: TF-IDF + Logistic Regression (multi-class)
- **Sentiment Classifier**: TF-IDF + Logistic Regression (4-class: Critical/Negative/Neutral/Positive)
- **API**: FastAPI with `/predict` endpoint

## 📁 Repository Structure
```
├── Week1_EDA_Grievance_NLP.ipynb
├── Week2_Classification.ipynb
├── Week3_SentimentAnalysis.ipynb
├── Week4_API_Evaluation_FinalDelivery.ipynb
├── main.py                    ← FastAPI app
├── models/                    ← gitignored — model weights
├── .gitignore
└── README.md
```

## 🚀 How to Run
```bash
pip install -r requirements.txt
uvicorn main:app --reload
# Open http://127.0.0.1:8000/docs
```

## 📊 Model Performance
| Model | Accuracy | Macro F1 |
|-------|----------|----------|
| Department Classifier | -- | -- |
| Sentiment Classifier  | -- | -- |

> Fill in your actual scores after running Week 4 notebook.

## 🔗 API Usage
```bash
curl -X POST http://127.0.0.1:8000/predict \\
  -H 'Content-Type: application/json' \\
  -d '{"complaint_text": "Fraudulent charges on my credit card, this is urgent!"}'
```
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print('✅ README.md created!')

In [ ]:
print('\n' + '='*60)
print('🎉 WEEK 4 COMPLETE — FINAL CHECKLIST')
print('='*60)

checklist = [
    ('Department classifier trained & evaluated', True),
    ('Confusion matrix generated', True),
    ('Classification report printed', True),
    ('Sentiment classifier trained & evaluated', True),
    ('Macro F1-score computed', True),
    ('Priority score logic implemented', True),
    ('Models serialized with joblib', True),
    ('FastAPI app (main.py) written', True),
    ('API tested with sample complaints', True),
    ('.gitignore created', True),
    ('README.md written', True),
    ('4 weeks of GitHub commits', False),  # manually verify
]

for item, done in checklist:
    icon = '✅' if done else '⬜'
    print(f'  {icon} {item}')

print('\n🚀 Suggested final commit messages:')
print('  deploy: serialized dept and sentiment models with joblib')
print('  deploy: built FastAPI /predict endpoint with priority scoring')
print('  eval: generated confusion matrices and classification reports')
print('  docs: added README.md and .gitignore for final submission')